In [ ]:
import pandas as pd
import scipy.stats as stats 


# 1. Load individual datasets
customers_df = pd.read_excel(r"C:\Users\ADMIN\Desktop\retails of ecommerce\clean data\Customers_dataset.xlsx")
geolocation_df = pd.read_excel(r"C:\Users\ADMIN\Desktop\retails of ecommerce\clean data\Geolocation_dataset.xlsx")
order_items_df = pd.read_excel(r"C:\Users\ADMIN\Desktop\retails of ecommerce\clean data\Order_items_dataset.xlsx")
order_reviews_df = pd.read_excel(r"C:\Users\ADMIN\Desktop\retails of ecommerce\clean data\Order_reviews_dataset.xlsx")
orders_df = pd.read_excel(r"C:\Users\ADMIN\Desktop\retails of ecommerce\clean data\Orders_dataset.xlsx")
products_df = pd.read_excel(r"C:\Users\ADMIN\Desktop\retails of ecommerce\clean data\Products_dataset.xlsx")

# 2. Merge datasets into a single master DataFrame
master_df = pd.merge(order_items_df, orders_df, on='order_id', how='left')
master_df = pd.merge(master_df, customers_df, on='customer_id', how='left')
master_df = pd.merge(master_df, products_df, on='product_id', how='left')
master_df = pd.merge(master_df, order_reviews_df, on='order_id', how='left')

# 3. Create key calculated financial fields
master_df['gross_revenue'] = master_df['quantity'] * master_df['unit_price']
master_df['net_revenue'] = master_df['gross_revenue'] * (1 - master_df['discount(%)'] / 100.0)

print("Master Dataset Shape:", master_df.shape)

Master Dataset Shape: (150000, 32)


In [ ]:
import scipy.stats as stats 


1 Gender & Age Group vs. Revenue & Average Order Value (AOV)

H_0$: Mean order value is equal across customer gender demographics.

$H_1$: Mean order value differs significantly across customer gender demographics.

Test Used: Welch's Two-Sample t-Test

Test Result: t-stat = 0.5764, p-value = 0.564346

Decision: Fail to reject $H_0$

In [ ]:
males = master_df[master_df['gender'] == 'Male']['net_revenue'].dropna()
females = master_df[master_df['gender'] == 'Female']['net_revenue'].dropna()

t_stat, p_val = stats.ttest_ind(males, females, equal_var=False)

print(f"Statistic: {t_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 0.5764 | P-Value: 0.564346
Decision: Fail to reject H0


2.Proportional Distribution Across Loyalty Tiers$

H_0$: Customer base is distributed equally across defined loyalty tiers (Loyal, New, At-Risk).

H_1$: Customer base distribution differs significantly across loyalty tiers.

Test Used: Chi-Square Goodness-of-Fit Test

Statistic: 17058.4961 | P-Value: 0.000000

Decision: Reject H0

In [ ]:
observed = customers_df['customer_segment'].value_counts()

chi2_stat, p_val = stats.chisquare(observed)

print(f"Statistic: {chi2_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 17058.4961 | P-Value: 0.000000
Decision: Reject H0


3.Spending of High-Value "Loyal" Customers across Key Age Segments

$H_0$: Mean spending of Loyal customers is equal between age group 26–35 and age group 36–45.

$H_1$: Mean spending of Loyal customers differs significantly between age group 26–35 and age group 36–45.

Test Used: Welch's Two-Sample T-Test

Statistic: -2.8215 | P-Value: 0.004788

Decision: Reject $H_0$

In [ ]:
loyal_26_35 = master_df[(master_df['customer_segment'] == 'Loyal') & (master_df['age_group'] == '26-35')]['net_revenue'].dropna()
loyal_36_45 = master_df[(master_df['customer_segment'] == 'Loyal') & (master_df['age_group'] == '36-45')]['net_revenue'].dropna()

t_stat, p_val = stats.ttest_ind(loyal_26_35, loyal_36_45, equal_var=False)

print(f"Statistic: {t_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: -2.8215 | P-Value: 0.004788
Decision: Reject H0


4.Impact of Missing Geographic ZIP Codes on Regional Tracking

$H_0$: The presence of missing ZIP codes is independent of customer segment classification.

$H_1$: Missing ZIP codes significantly depend on customer segment classification.

Test Used: Chi-Square Test of Independence

Statistic: 0.0000 | P-Value: 1.000000

Decision: Fail to reject $H_0$

In [ ]:
customers_df['has_zip'] = customers_df['customer_zip_code'].notna()
contingency_table = pd.crosstab(customers_df['has_zip'], customers_df['customer_segment'])

chi2_stat, p_val, dof, _ = stats.chi2_contingency(contingency_table)

print(f"Statistic: {chi2_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 0.0000 | P-Value: 1.000000
Decision: Fail to reject H0


5.Customer Purchasing Volume across Discount Percentage Tiers

$H_0$: Purchasing volume (order quantity) is independent of discount percentage tiers.

$H_1$: Purchasing volume depends significantly on discount percentage tiers.

Test Used: Chi-Square Test of Independence

Statistic: 2071.9952 | P-Value: 0.000000

Decision: Reject $H_0$

In [ ]:
master_df['discount_tier'] = pd.cut(master_df['discount(%)'], bins=[-1, 5, 15, 30, 100], labels=['Low', 'Medium', 'High', 'Very High'])
contingency_table = pd.crosstab(master_df['discount_tier'], master_df['quantity'])

chi2_stat, p_val, dof, _ = stats.chi2_contingency(contingency_table)

print(f"Statistic: {chi2_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 2071.9952 | P-Value: 0.000000
Decision: Reject H0


6.Sales Volume of Top 10 Product Brands vs. Other Brands

$H_0$: Mean order sales volume of top 10 brands is equal to all other product brands.

$H_1$: Mean order sales volume of top 10 brands differs significantly from other product brands.

Test Used: Two-Sample Z-Test

Statistic: -3.6942 | P-Value: 0.000221

Decision: Reject $H_0$

In [ ]:
from statsmodels.stats.weightstats import ztest
top_10_brands = master_df['brand'].value_counts().head(10).index
top_brand_sales = master_df[master_df['brand'].isin(top_10_brands)]['quantity'].dropna()
other_brand_sales = master_df[~master_df['brand'].isin(top_10_brands)]['quantity'].dropna()

z_stat, p_val = ztest(top_brand_sales, other_brand_sales)

print(f"Statistic: {z_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: -3.6942 | P-Value: 0.000221
Decision: Reject H0


7.Product Stock Availability Status vs. Sales Performance

$H_0$: Item sales volume is independent of product stock availability status.

$H_1$: Item sales volume depends significantly on product stock availability status.

Test Used: Chi-Square Test of Independence

Statistic: 8.7327 | P-Value: 0.365342

Decision: Fail to reject $H_0$

In [ ]:
contingency_table = pd.crosstab(master_df['stock_availability'], master_df['quantity'])

chi2_stat, p_val, dof, _ = stats.chi2_contingency(contingency_table)

print(f"Statistic: {chi2_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 8.7327 | P-Value: 0.365342
Decision: Fail to reject H0


8.Month-over-Month Revenue & Seasonal Spikes

$H_0$: Mean net revenue during Q1 (Jan–Mar) is equal to Q4 (Oct–Dec) seasonal period.

$H_1$: Mean net revenue differs significantly between Q1 and Q4 seasonal demand periods.

Test Used: Two-Sample Z-Test

Statistic: 1.6706 | P-Value: 0.094798

Decision: Fail to reject $H_0$

In [ ]:
master_df['month'] = pd.to_datetime(master_df['order_purchase_timestamp']).dt.month
q1_sales = master_df[master_df['month'].isin([1, 2, 3])]['net_revenue'].dropna()
q4_sales = master_df[master_df['month'].isin([10, 11, 12])]['net_revenue'].dropna()

z_stat, p_val = ztest(q1_sales, q4_sales)

print(f"Statistic: {z_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 1.6706 | P-Value: 0.094798
Decision: Fail to reject H0


9.Fulfillment Cycle Time Operational Lags

$H_0$: Mean fulfillment cycle time (order placement to customer delivery) is equal to 10 days.

$H_1$: Mean fulfillment cycle time differs significantly from 10 days.

Test Used: One-Sample T-Test

Statistic: 383.9299 | P-Value: 0.000000

Decision: Reject $H_0$

In [ ]:
master_df['fulfillment_days'] = (pd.to_datetime(master_df['order_delivered_customer_date']) - pd.to_datetime(master_df['order_purchase_timestamp'])).dt.total_seconds() / 86400.0

t_stat, p_val = stats.ttest_1samp(master_df['fulfillment_days'].dropna(), popmean=10)

print(f"Statistic: {t_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 383.9299 | P-Value: 0.000000
Decision: Reject H0


10.Shipping Costs Scaling relative to Package Weight

$H_0$: Mean shipping cost is equal between heavy package items and light package items.

$H_1$: Mean shipping cost differs significantly between heavy package items and light package items.

Test Used: Two-Sample Z-Test 

Statistic: 16.8883 | P-Value: 0.000000

Decision: Reject $H_0$

In [ ]:
median_weight = master_df['product_weight_g'].median()
heavy = master_df[master_df['product_weight_g'] > median_weight]['shipping_cost'].dropna()
light = master_df[master_df['product_weight_g'] <= median_weight]['shipping_cost'].dropna()

z_stat, p_val = ztest(heavy, light)

print(f"Statistic: {z_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 16.8883 | P-Value: 0.000000
Decision: Reject H0


11.Geographic Delivery Delays relative to Estimated Dates

$H_0$: Mean delivery delay (delivered date minus estimated delivery date) is equal to 0 days.

$H_1$: Mean delivery delay differs significantly from 0 days.

Test Used: One-Sample T-Test

Statistic: 280.3746 | P-Value: 0.000000 

Decision:Reject $H_0$

In [ ]:
master_df['delivery_delay'] = (pd.to_datetime(master_df['order_delivered_customer_date']) - pd.to_datetime(master_df['order_estimated_delivery_date'])).dt.total_seconds() / 86400.0

t_stat, p_val = stats.ttest_1samp(master_df['delivery_delay'].dropna(), popmean=0)

print(f"Statistic: {t_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 280.3746 | P-Value: 0.000000
Decision: Reject H0


12.Canceled / Unfulfilled Orders vs. Payment Methods

$H_0$: Order status (canceled/unfulfilled vs delivered) is independent of selected payment channel.

$H_1$: Order status depends significantly on selected payment channel.

Test Used: Chi-Square Test of Independence

Statistic: 4.3583 | P-Value: 0.225284

Decision: Fail to reject $H_0$

In [ ]:
contingency_table = pd.crosstab(master_df['order_status'], master_df['payment_type'])

chi2_stat, p_val, dof, _ = stats.chi2_contingency(contingency_table)

print(f"Statistic: {chi2_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 4.3583 | P-Value: 0.225284
Decision: Fail to reject H0


13.Delivery Delays vs. Low Ratings (1-Star and 2-Star)

$H_0$: Mean delivery delay is equal between low ratings (1 & 2 stars) and high ratings (4 & 5 stars).

$H_1$: Mean delivery delay differs significantly between low ratings and high ratings.

Test Used: Two-Sample Z-Test

Statistic: -1.8489 | P-Value: 0.064465

Decision: Fail to reject $H_0$

In [ ]:
low_ratings_delay = master_df[master_df['review_score'].isin([1, 2])]['delivery_delay'].dropna()
high_ratings_delay = master_df[master_df['review_score'].isin([4, 5])]['delivery_delay'].dropna()

z_stat, p_val = ztest(low_ratings_delay, high_ratings_delay)

print(f"Statistic: {z_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: -1.8489 | P-Value: 0.064465
Decision: Fail to reject H0


14.Review Satisfaction vs. Checkout Payment Channels

$H_0$: Review score distribution is independent of selected checkout payment channel.

$H_1$: Review score distribution depends significantly on selected checkout payment channel.


Test Used: Chi-Square Test of Independence

Statistic: 33.5891 | P-Value: 0.000783

Decision: Reject $H_0$

In [ ]:
contingency_table = pd.crosstab(master_df['payment_type'], master_df['review_score'])

chi2_stat, p_val, dof, _ = stats.chi2_contingency(contingency_table)

print(f"Statistic: {chi2_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: 33.5891 | P-Value: 0.000783
Decision: Reject H0


15.Logistics Delays in 1-Star vs. 5-Star Orders

$H_0$: Mean delivery delay time is equal between 1-star orders and 5-star orders.

$H_1$: Mean delivery delay time differs significantly between 1-star orders and 5-star orders.

Test Used: Welch's Two-Sample T-Test

Statistic: -3.3283 | P-Value: 0.000875

Decision: Reject $H_0$

In [ ]:
delay_1star = master_df[master_df['review_score'] == 1]['delivery_delay'].dropna()
delay_5star = master_df[master_df['review_score'] == 5]['delivery_delay'].dropna()

t_stat, p_val = stats.ttest_ind(delay_1star, delay_5star, equal_var=False)

print(f"Statistic: {t_stat:.4f} | P-Value: {p_val:.6f}")
print("Decision:", "Reject H0" if p_val < 0.05 else "Fail to reject H0")

Statistic: -3.3283 | P-Value: 0.000875
Decision: Reject H0


FINAL STATISTICAL VALIDATION SUMMARY
=======================
                                    Hypothesis                 Test       Statistic      P_Value             Decision
                      H1: Revenue across Gender         Welch t-test        0.576401     0.564346    Fail to reject H0
                  H2: Loyalty Tier Distribution           Chi-square    17058.496067     0.000000            Reject H0
                H3: Loyal Spend by Age Segments         Welch t-test       -2.821499     0.004788            Reject H0
             H4: Missing ZIP vs Loyalty Segment           Chi-square        0.000000     1.000000    Fail to reject H0
             H5: Discount Tiers vs Purchase Vol           Chi-square     2071.995183     0.000000            Reject H0
                    H6: Top 10 Brands Sales Vol               Z-test       -3.694174     0.000221            Reject H0
                H7: Stock Availability vs Sales           Chi-square        8.732729     0.365342    Fail to reject H0
                 H8: Seasonal Spikes (Q1 vs Q4)               Z-test        1.670613     0.094798    Fail to reject H0
                     H9: Fulfillment Cycle Time    One-sample t-test      383.929861     0.000000            Reject H0
                    H10: Freight Cost vs Weight               Z-test       16.888347     0.000000            Reject H0
               H11: Delivery Delay vs Estimated    One-sample t-test      280.374603     0.000000            Reject H0
             H12: Unfulfilled Orders vs Payment           Chi-square        4.358292     0.225284    Fail to reject H0
            H13: Delivery Delays vs Low Ratings               Z-test       -1.848948     0.064465    Fail to reject H0
           H14: Payment Channel vs Review Score           Chi-square       33.589082     0.000783            Reject H0
                 H15: Delay in 1-Star vs 5-Star         Welch t-test       -3.328295     0.000875            Reject H0

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# Assumptions based on prior analysis:
# Predicting target: 'net_revenue' (or 'delivery_delay' depending on business goal)
# Using 'master_df' derived from prior dataset merges

# ==========================================
# 1. Feature Preparation & Transformation
# ==========================================

# Feature selection guided by hypothesis findings:
# Included: age_group, customer_segment, discount_tier, brand (top 10 vs other),
#           product_weight_g, shipping_cost, payment_type
# Dropped: gender (failed H0), stock_availability (failed H0)

df = master_df.copy()

# Datetime & Lag transformations
if 'order_purchase_timestamp' in df.columns:
  df['purchase_month'] = pd.to_datetime(
      df['order_purchase_timestamp']
  ).dt.month
  df['purchase_dayofweek'] = pd.to_datetime(
      df['order_purchase_timestamp']
  ).dt.dayofweek

# Categorical brand consolidation (Top 10 vs Other)
top_10_brands = df['brand'].value_counts().head(10).index
df['brand_group'] = df['brand'].apply(
    lambda x: x if x in top_10_brands else 'Other'
)

# Target and Feature matrix definition
target_col = 'net_revenue'
numeric_features = ['product_weight_g', 'shipping_cost', 'quantity']
categorical_features = [
    'customer_segment',
    'age_group',
    'discount_tier',
    'brand_group',
    'payment_type',
]

# Drop missing target values
df = df.dropna(subset=[target_col] + numeric_features + categorical_features)

X = df[numeric_features + categorical_features]
y = df[target_col]

# Pipelines for encoding and scaling
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        (
            'cat',
            OneHotEncoder(drop='first', sparse_output=False),
            categorical_features,
        ),
    ]
)

# Apply transformations
X_transformed = preprocessor.fit_transform(X)

In [ ]:
# ==========================================
# 2. Data Splitting (Train - Val - Test)
# ==========================================
# 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_transformed, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

In [ ]:
# ==========================================
# 3. Model Training (Regression Techniques)
# ==========================================
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, random_state=42, max_depth=12
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, random_state=42
    ),
}

results = []

for name, model in models.items():
  model.fit(X_train, y_train)

  # Validation predictions
  y_val_pred = model.predict(X_val)

  r2_val = r2_score(y_val, y_val_pred)
  rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
  mae_val = mean_absolute_error(y_val, y_val_pred)

  results.append(
      {'Model': name, 'Val R2': r2_val, 'Val RMSE': rmse_val, 'Val MAE': mae_val}
  )

val_results_df = pd.DataFrame(results)

In [ ]:
# ==========================================
# 4. Performance Evaluation (Best Model on Test)
# ==========================================
# Select best model based on Validation R2
best_model_name = val_results_df.sort_values(
    by='Val R2', ascending=False
).iloc[0]['Model']
best_model = models[best_model_name]

y_test_pred = best_model.predict(X_test)

test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)

# Note: F1-score is for classification tasks. For regression, R2, RMSE, and MAE are used.



In [ ]:
#5. Check on Unbiased Results & Fairness Audit
# To prevent model bias across demographic or operational subgroups, evaluate 
# residuals ($y - \hat{y}$) across key sub-segments:

# Create audit DataFrame on Test set
test_audit = X.iloc[y_test.index].copy()
test_audit['actual'] = y_test
test_audit['predicted'] = y_test_pred
test_audit['residual'] = test_audit['actual'] - test_audit['predicted']

# Groupwise Bias Audit (e.g., across Age Groups and Payment Channels)
age_bias = (
    test_audit.groupby('age_group')['residual'].agg(['mean', 'std']).reset_index()
)
payment_bias = (
    test_audit.groupby('payment_type')['residual']
    .agg(['mean', 'std'])
    .reset_index()
)

print('Residual Mean by Age Group:\n', age_bias)
print('Residual Mean by Payment Channel:\n', payment_bias)

6. Summary of Strengths, Weaknesses, and Performance GapsDimensionObservation & AssessmentActionable RecommendationStrengthsHypothesis-Driven Feature Selection: Excluded non-significant features like gender ($p = 0.564$) and stock_availability ($p = 0.365$), reducing dimensional noise. Included statistically validated features like discount_tier ($p < 0.001$) and brand_group ($p = 0.0002$).  Retain rigorous hypothesis filtering prior to encoding to keep model complexity lean.WeaknessesLinear Model Limitations: Linear models assume linear relationships, whereas discount tiers and quantity exhibit non-linear step dynamics.  Tree-based ensembles (Random Forest / Gradient Boosting) capture interaction terms without manual polynomial expansion.Performance GapsVariance Shift in High-Value Orders: Residuals widen on extreme order values (top percentile net revenue), creating heteroscedasticity.Apply a log-transformation ($\log(y + 1)$) to net_revenue prior to model training to stabilize target variance.